# Regression Model Notebook

Earlier regression model-selection and diagnostics using full_rent_features.csv.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

## Load Processed Dataset

In [ ]:
features = pd.read_csv("full_rent_features.csv")
features["log_median_rent_psq"] = np.log(features["median_rent_psq"])

## Candidate Model Formulas

In [ ]:
from itertools import combinations

feature_cols = features.drop(
    columns=["lat", "lon", "median_rent", "median_rent_psq", "listing_count", "log_median_rent_psq"]
).columns

formulas = []
log_formulas = []

for i in range(len(feature_cols) + 1):
    for subset in combinations(feature_cols, i):
        if len(subset) == 0:
            formulas.append("median_rent_psq ~ 1")
            log_formulas.append("log_median_rent_psq ~ 1")
        else:
            formulas.append(f"median_rent_psq ~ {' + '.join(subset)}")
            log_formulas.append(f"log_median_rent_psq ~ {' + '.join(subset)}")

## AIC Model Selection

In [ ]:
models = []
log_models = []

for formula in formulas:
    models.append(smf.ols(formula=formula, data=features).fit().aic)

for formula in log_formulas:
    log_models.append(smf.ols(formula=formula, data=features).fit().aic)

print(f"Best regular model is {formulas[np.argmin(models)]} with AIC: {np.min(models)}")
print(f"Best regular model is {log_formulas[np.argmin(log_models)]} with AIC: {np.min(log_models)}")

## BIC Model Selection

In [ ]:
models = []
log_models = []

for formula in formulas:
    models.append(smf.ols(formula=formula, data=features).fit().bic)

for formula in log_formulas:
    log_models.append(smf.ols(formula=formula, data=features).fit().bic)

print(f"Best regular model is {formulas[np.argmin(models)]} with BIC: {np.min(models)}")
print(f"Best regular model is {log_formulas[np.argmin(log_models)]} with BIC: {np.min(log_models)}")

## Selected Regression Models

In [ ]:
model1 = smf.ols(
    formula="log_median_rent_psq ~ shores_beach_ft + utc_ft + nearest_park_ft", data=features
).fit()

model2 = smf.ols(
    formula="log_median_rent_psq ~ shores_beach_ft + utc_ft", data=features
).fit()

print(model1.summary())
print(model2.summary())

## Multicollinearity Check

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def vif_calculator(model):
    X = model.model.exog
    names = model.model.exog_names

    vif = pd.DataFrame({
        "Variable": names,
        "VIF": [variance_inflation_factor(X, i)
                for i in range(X.shape[1])]
    })

    return vif

print(vif_calculator(model1))
print("\n")
print(vif_calculator(model2))

## Regression Diagnostics

In [ ]:
import matplotlib.pyplot as plt

fitted = model1.fittedvalues
residuals = model1.resid

plt.figure(figsize=(6, 4))
plt.scatter(fitted, residuals)
plt.title("Residuals vs Fitted Value")
plt.xlabel("Fitted Value (dollars per square foot per month)")
plt.ylabel("Residuals")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sm.qqplot(model1.resid, line="s", ax=ax)
plt.title("QQ Plot of Residuals");

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(features["log_median_rent_psq"])

## UCSD Housing Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
features["median_rent_psq"].hist(ax=ax, bins=20)
plt.axvline(x=7.51, color="red", label="On-Campus Housing")
plt.title("Histogram of Median Rent Per Square Foot")
plt.xlabel("Price Per Square Foot Per Month")
plt.legend(loc="upper right", bbox_to_anchor=(0.95, 1));

## Prediction Interval

In [ ]:
pred_df = pd.DataFrame({
    "shores_beach_ft": [12038],
    "utc_ft": [10616],
    "nearest_park_ft": [436] 
})

pred = model1.get_prediction(pred_df)
summary = pred.summary_frame(alpha=0.01)
print(summary)